# LOG05 | 미션 01 | Student Workbook

## 이번 미션에서 하는 일

**무인 보급장비(수송 드론, 무인 자율주행 차량) 운용 환경 및 사고 위험 요인 분석**

1. 실습 목적 및 개요 역할 및 배경: 학습자는 무인 보급장비 투입 전 상태·책임·중지조건 체크리스트 교육용 실습의 참여자로서 임무를 수행합니다. 핵심 과제: 수송 드론과 무인 지상차량이 운용되는 가상 환경의 특성과 사고 위험 요인을 분류하고, 체크리스트 문서의 전체 골격을 정의합니다. 2. 주의 및 검증 지침 데이터 해석의 엄정성: 자료나 AI 설명을 철저히 검증하지 않으면 미확인 기록이나 모의 회신을 실제 현장 확인이나 승인으로 오인할 수 있으므로 주의해야 합니다. 수행 방식: 본인 판단, AI 초안 및 역할 대화, 근거 대조 및 보완, 최종 기록의 전 과정을 원본 자료 범위 내에서 수행합니다. 3. 산출물 및 소요 시간 안내 결과 파일 목록: docs/log05_doc_framework.md LOG05_M01_forms.md forms/LOG05_M01_forms.md 시간 안내: 원본 예상 시간은 1.5시간이며, CMS 편성은 원본 durationHours 를 올림하여 2시간으로 구성되어 있습니다 (실제 소요 시간은 실습 완료 후 확인).

**학습 목표:** 가상 자료를 읽고 자신의 최초 판단을 적은 다음, AI 초안을 원자료와 대조하여 문서와 검증 근거를 완성합니다. 실제 장비 제어·비행·파괴·무력화 방법을 설계하거나 실행하는 훈련은 아닙니다.

**준비물:** 이 Notebook, 미션 1의 가상 자료 및 빈 작성 양식, 기관이 허용한 AI 환경(가능한 경우).

**학습 환경:** Colab은 인터넷이 필요합니다. 폐쇄망에서는 사전 설치된 Jupyter 또는 별도 Markdown 양식으로 진행할 수 있으며 외부 AI를 사용할 수 없으면 미실시로 기록하세요.

**학습 순서:** 자료 먼저 읽기 → 본인 생각 기록 → AI 질문 → 원문 근거 대조 → 수정 → 저장·제출.

**구분:** 파일 저장 / 실습 수행 / 평가 기준 충족 / 실제 교관 승인은 각각 다른 상태입니다.

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 원본 가상 자료와 작성 양식

아래는 첨부된 **LOG05 실습자료 원본**에서 미션별로 추출한 자료입니다. 모의 기록에서 아직 공개되지 않은 카드나 결과는 미리 확정하지 마세요.

미션 1. 무인 보급장비(수송 드론, 무인 자율주행 차량) 운용 환경 및 사고 위험 요인 분석

**가. 제공 자료 1 - 가상 운용 환경표 [교육용가상자료]**

| 구간 | 지형 | 통신 | 기상 특성 | 주로 운용하는 장비 |
| --- | --- | --- | --- | --- |
| A | 평지, 포장 도로 인접 | 링크 품질 안정 | 대체로 양호 | UV-2 |
| B | 숲 인접, 굴곡 | 통신 음영 가능(링크 일시 두절) | 안개 잦음 | UV-2, UD-1 |
| C | 개활지 | 링크 품질 안정, 거리가 멀면 저하 | 강풍 가능 | UD-1 |

**가. 제공 자료 2 - 가상 사고·위험 사례 5건 [교육용가상자료]**

| 사례 | 상황 요약 | 사고 전 점검·승인 | 눈에 띄는 한계 |
| --- | --- | --- | --- |
| C1 | UD-1이 배터리 잔량 40%로 출발했고 이륙 직후 급감해 도착 전 강제 착륙 | 잔량 표시만 확인 | 부하 시 전압 이상 확인 항목 없음 |
| C2 | UV-2가 B 구간에서 링크가 18초 두절된 뒤 경로를 이탈 | 경로 사전 입력 확인 | 두절 시 정지 규칙과 음영 구간 고지가 없음 |
| C3 | UD-1이 적재 한도 8단위를 넘겨 9단위로 출발해 이륙이 불안정 | 적재 담당 자기 신고 | 저울 기록 없음 |
| C4 | C 구간 풍속 등급 3 예보에도 출발 승인, 중간 회항 | 예보 확인 여부 기재 없음 | 풍속 기준이 문서에 없음 |
| C5 | 승인권자가 점검표 서명 없이 구두 승인한 임무에서 사고 발생 | 구두 승인 | 책임 소재를 문서로 가릴 수 없음 |

**나. 작성 서식 1 - 위험 요인 분류표(6개 분류)**

| 분류 | 위험 요인(사례에서 찾은 것) | 사고로 이어지는 경로 | 투입 전 확인할 것 |
| --- | --- | --- | --- |
| 장비 |  |  |  |
| 통신 |  |  |  |
| 적재 |  |  |  |
| 기상 |  |  |  |
| 사람 |  |  |  |
| 절차 |  |  |  |

**나. 작성 서식 2 - 사례별 빠진 점검·승인**

| 사례 | 있었던 점검·승인 | 빠진 점검·승인 | 추가로 확인할 것 |
| --- | --- | --- | --- |
| C1 |  |  |  |
| C2 |  |  |  |
| C3 |  |  |  |
| C4 |  |  |  |
| C5 |  |  |  |

**나. 작성 서식 3 - 지침서 목차 골격(지휘관 결심 의제 순으로 배열)**

| 순서 | 장 제목 | 담을 핵심 내용 | 지휘관이 이 장에서 얻는 판단 정보 |
| --- | --- | --- | --- |
| 제1장 |  |  |  |
| 제2장 |  |  |  |
| 제3장 |  |  |  |
| 제4장 |  |  |  |
| 제5장 |  |  |  |
| 제6장 |  |  |  |
| 제7장 |  |  |  |
| 제8장 |  |  |  |

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 가상 사례 5건을 모두 검토했다 | (   ) |
| 위험 요인을 장비·통신·적재·기상·사람·절차 6개 분류에 빠짐없이 나누었다 | (   ) |
| 사례마다 빠진 점검 또는 승인을 하나 이상 찾았다 | (   ) |
| 지침서 목차 8개 장의 제목과 핵심 내용을 정의했다 | (   ) |
| 목차를 지휘관이 먼저 알아야 할 의제 순으로 배열하고 이유를 한 줄로 적었다 | (   ) |

## 단계별 수행 안내와 미션 전용 AI 활용

### 활동 1. 본인이 사례와 질문을 정한다

**무엇을 하나요?** 1. 단계 개요 목표: 가상 보급로의 운용 환경(구간 A～C)과 가상 사고·위험 사례 5건( C1 ～ C5 )을 면밀히 검토하여, 운용 환경별 특성과 각 사고에서 드러난 문제점을 명확히 파악하고 AI에게 질의할 구체적인 탐색 질문을 스스로 수립합니다. 수행 방식: 막연한 독서나 단순 암기에 그치지 않고, 가상 환경의 제약 사항과 실제 사고 사례 간의 인과관계를 철저히 분석하여 본인 주도적으로 학습 및 분석 방향을 설정합니다. 2. 세부 수행 절차 가상 운용 환경 분석 (구간 A～C): 평지 및 포장도로 인접 구간(A), 숲 인접 및 통신 음영/안개 빈발 구간(B), 개활지 및 강풍 가능 구간(C)의 지형, 통신, 기상 특성을 파악하고 주로 운용되는 장비( UD-1 , UV-2 )와의 연계성을 분석합니다. 가상 사고·위험 사례 분석 (C1～C5): 배터리 잔량 오판 및 부하 전압 미확인( C1 ), 통신 두절 시 정지 규칙 부재로 인한 경로 이탈( C2 ), 저울 기록 없는 자기 신고 적재 초과( C3 ), 풍속 기준 문서 부재에 따른 기상 미숙 ( C4 ), 서명 없는 구두 승인으로 인한 책임 소재 불명확( C5 ) 사례를 각각 검토합니다. 확인된 사실 및 누락 기록 정리: 각 사례별로 당시 존재했던 점검·승인 내용과 결정적으로 누락된 항목(점검 항목, 문서 기준, 승인 서명 등)을 대비하여 기록합니다. AI 질의용 질문 수립: 식별된 위험 요인과 누락 기록을 바탕으로, 후속 단계(AI와 분류표·목차 작성 등)에서 심층적으로 검토할 핵심 탐색 질문을 작성합니다. 3. 주의 및 검증 사항 제공된 교육용 가상 자료( A～C , C1～C5 )의 내용에 근거하여 사실 관계를 파악해야 하며, 임의로 실제 

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 2. AI와 분류표·목차를 만든다

**무엇을 하나요?** 1. 단계 개요 목표: 앞서 1단계에서 분석한 가상 보급로의 운용 환경 및 사고·위험 사례 5건( C1 ～ C5 )을 바탕으로, AI 에이전트와 협업하여 6개 위험 분류(장비, 통신, 적재, 기상, 사람, 절차) 체계를 수립하고 체크리스트 지침서의 8개 장 목차 초안 을 도출하여 첫 답변을 안전하게 보존합니다. 수행 방식: AI의 답변을 무조건 수용하지 않고, 제공된 가상 사고 사례와 운용 환경 제약 사항이 분류표와 목차 골격에 충실히 반영되도록 구체적인 프롬프트를 통해 초안을 도출하고 기록합니다. 2. 세부 수행 절차 6개 위험 분류 체계 정립: 식별된 위험 요인을 장비, 통신, 적재, 기상, 사람, 절차의 6가지 영역으로 나누고, 각 영역별 주요 위험 요인과 사고로 이어지는 경로를 AI와 함께 체계적으로 구조화합니다. 8개 장 목차 초안 작성: 지휘관이 의사결정 시 가장 먼저 파악해야 할 의제 순서에 발맞추어, 체크리스트 지침서의 8개 장 목차 골격을 생성하고 각 장에 담길 핵심 내용과 기대 효과를 정리합니다. 첫 답변 보존 및 기록: AI 에이전트가 제시한 초기 분석 결과와 목차 초안의 첫 답변 내용을 누락 없이 보존하고, 후속 단계(사실 및 원인 단정 검토)에서 대조할 수 있도록 준비합니다. 3. 주의 및 검증 사항 AI가 제안하는 분류표나 목차가 추상적이거나 실제 가상 사고 사례( C1 ～ C5 )와 연계되지 않는 부분이 없는지 점검해야 합니다. 도출된 초안은 최종 확정본이 아니며, 다음 단계의 엄밀한 근거 대조와 교차 검증을 위한 기초 자료로 활용되도록 명확히 보존해야 합니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 3. 사실과 원인 단정을 검토한다

**무엇을 하나요?** 1. 단계 개요 목표: AI 에이전트가 제시한 분석 내용 중 핵심 주장 3건 이상을 원자료(가상 사고 사례 및 운용 환경표)와 철저히 대조하고, 특히 적재 계산 방식 및 구두 승인 해석 과정에서 발생할 수 있는 오류를 수정·재확인합니다. 수행 방식: AI의 답변을 맹신하지 않고, 제공된 가상 자료의 세부 수치와 절차적 한계를 직접 대조하여 부당한 원인 단정이나 왜곡된 해석을 바로잡습니다. 2. 세부 수행 절차 AI 주장 3건 이상 선정 및 원자료 대조: AI가 도출한 위험 요인 분류나 사고 원인 분석 중 대표적인 주장 3건 이상을 추출하여, 가상 사고 사례(C1～C5) 및 운용 환경 데이터와 일대일로 비교합니다. 적재 계산 및 자기 신고 한계 검토: C3 사례(저울 기록 없는 자기 신고 및 적재 초과)와 관련하여, AI가 적재 상태나 중량을 임의로 추정하거나 과소평가한 부분이 없는지 원자료의 기록 부재 사실과 대조하여 검증합니다. 구두 승인 해석 및 책임 소재 검토: C5 사례(서명 없는 구두 승인)와 관련하여, 구두 승인이 현장 조치로 정당화될 수 있다는 식의 잘못된 해석을 바로잡고 문서화된 승인 절차 부재가 초래하는 책임 소재의 한계를 명확히 재확인합니다. 해석 수정 및 최종 검증 기록: 대조 과정에서 발견된 오류나 과장된 해석을 바로잡고, 수정된 분석 내용을 실습 기록 서식에 반영합니다. 3. 주의 및 검증 사항 실제 현장 확인이나 공식 기록이 부재함에도 불구하고 AI의 설명만으로 사고 원인이 완전히 규명된 것처럼 단정하여 기록하지 않도록 주의해야 합니다. 사실과 추정, 그리고 교육용 가상 자료에서 명시된 내용과 학습자의 해석 영역을 명확히 구분하여 서식에 기재해야 합니다.

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

### 활동 4. 분석 결과와 의제 순서를 넘긴다

**무엇을 하나요?** 1. 단계 개요 목표: 미션 1을 통해 도출한 가상 사고·위험 사례 5건 검토 결과, 6개 위험 분류 체계, 8개 장 목차 및 지휘관 결심 의제 우선순위 이유, 그리고 분석 과정에서 확인된 미확인(불확실성) 요인들을 후속 미션인 미션 2(투입 전 점검 체크리스트 작성)와 미션 10(표준 지침서 최종 통합)으로 온전히 연계합니다. 수행 방식: 단편적인 개별 미션 수행에 머무르지 않고, 앞서 분석한 기초 골격과 식별된 한계점을 다음 단계의 체크리스트 검증 및 최종 지침서 편성에 직접 활용할 수 있도록 정돈된 형태로 넘깁니다. 2. 세부 수행 절차 M1 산출물 요약 및 인계 파일 정리: 가상 사고 사례 5건 검토 내용, 6개 위험 분류(장비·통신·적재·기상·사람·절차)표, 그리고 지휘관 의제 순서에 따른 8개 장 목차 골격을 최종 점검하여 인계 준비를 마칩니다. 우선순위 선정 근거 및 미확인 요인 명시: 지침서 목차 배치 시 지휘관이 가장 먼저 파악해야 할 의제로 설정한 이유와, 데이터 부족 등으로 인해 명확히 규명하지 못한 미확인(정보 공백) 요인들을 명확히 기록하여 후속 미션(M2, M3 등)에서 보완할 수 있도록 합니다. 후속 미션(M2 및 M10) 연계 지점 확인: 도출된 6개 위험 분류와 위험 요인들이 미션 2의 4개 점검 영역(배터리/유류, 통신, 적재, 기상) 체크리스트 항목 설계에 빠짐없이 반영되도록 연결 구조를 점검합니다. 3. 주의 및 검증 사항 미션 1의 분석 결과나 목차 뼈대를 단순히 형식적으로 넘기지 않고, 실제 미션 2와 미션 10의 산출물 작성에 실질적인 기초 자료로 활용될 수 있도록 구체적인 내용과 근거를 함께 전달해야 합니다. 검증되지 않은 가상 가정이나 추정을 마치 확정

**직접 물어볼 AI 프롬프트:** 제공 기록 초안·근거 대조·검산·레드팀·모의 역할 대화

**검토:** 원자료 번호를 표시하고 실제 AI 회신을 수정한 이유를 적으세요.

## 제출 전에 확인할 기준

**평가 항목 1:** [정량적 평가] [필수] 운용 환경·위험 요인 분석 — 가상 사례·문헌 5건 이상을 검토해 위험 요인을 6개 분류로 정리했는가, 사례별로 빠진 점검·승인을 찾아 기록했는가, 지침서 목차(8개 장) 골격을 100% 정의하고 지휘관 결심 의제 순으로 배열했는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 수송 드론과 무인 지상차량이 운용되는 가상 환경의 특성과 사고 위험 요인을 분류하고, 체크리스트 문서의 전체 골격을 정의한다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: docs/log05_doc_framework.md, LOG05_M01_forms.md, forms/LOG05_M01_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**중요:** 정답·실제 측정값·사람의 서명이나 승인을 AI가 만들어 주지 않도록 합니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m01", "steps": [{"title": "본인이 사례와 질문을 정한다"}, {"title": "AI와 분류표·목차를 만든다"}, {"title": "사실과 원인 단정을 검토한다"}, {"title": "분석 결과와 의제 순서를 넘긴다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)


## 마무리

작성 JSON과 검토용 ZIP을 내려받아 보관하세요. ZIP은 **검토용**이며 평가 PASS나 교관 승인 증명이 아닙니다. 전 단계 결과가 필요하면 본인의 실제 이전 기록을 사용하고, 기록이 없으면 **미실시**로 남기세요.